# 22 モメンタム仮説検証（日本株）

| 項目 | 内容 |
|------|------|
| プロジェクトID | STOCK-PRED-001 |
| 入力データ | `data/equity_jp_ohlcv.parquet`, `data/macro.parquet` |
| 作成者 | |
| 更新日 | 2026-07-08 |

## 検証する仮説

**帰無仮説（H₀）**: 過去リターンは将来リターンを予測しない（IC = 0）  
**対立仮説（H₁）**: 過去リターンは将来リターンと有意な相関を持つ（モメンタム効果）  
**有意水準（α）**: 0.05

## 背景
モメンタム効果（過去の勝ち銘柄が将来も勝ちやすい）は Jegadeesh & Titman (1993) 以来、
多くの市場で実証されてきた。日本株では米国と異なりモメンタムが弱い・逆転する期間があるとも
言われる。本ノートブックでは IC 分析・クインタイル分析・統計的頑健性検定を通じて
日本株のモメンタム特性を体系的に検証する。

## 結論メモ
<!-- 検証後に記入: p値・効果量・解釈・実務的含意 -->

---
## 0. 環境セットアップ

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import yaml

try:
    import japanize_matplotlib
except ImportError:
    print('japanize_matplotlib not available — Japanese labels may not render correctly')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 100)
%matplotlib inline

ALPHA = 0.05
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup OK')

In [ ]:
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR    = Path(cfg['paths']['data'])
FIGURES_DIR = Path(cfg['paths']['figures'])
TABLES_DIR  = Path(cfg['paths']['tables'])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data dir    : {DATA_DIR.resolve()}')
print(f'Figures dir : {FIGURES_DIR.resolve()}')

---
## 1. データ読み込み

In [ ]:
ohlcv_path = DATA_DIR / 'equity_jp_ohlcv.parquet'
macro_path = DATA_DIR / 'macro.parquet'

if not ohlcv_path.exists():
    raise FileNotFoundError(
        f'{ohlcv_path} が見つかりません。\n'
        '先に 01_data/03_equity_data_collection.ipynb を実行してデータを収集してください。'
    )

df_raw = pd.read_parquet(ohlcv_path)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values(['symbol', 'date']).reset_index(drop=True)

# マクロデータ（ニッケイ225）
df_macro = None
if macro_path.exists():
    df_macro = pd.read_parquet(macro_path)
    df_macro['date'] = pd.to_datetime(df_macro['date'])
    print(f'Macro loaded: {df_macro.shape}')
else:
    print('WARN: macro.parquet not found — market regime analysis will be limited')

print(f'OHLCV: {df_raw.shape[0]:,} rows  {df_raw["symbol"].nunique()} symbols')
print(f'Period: {df_raw["date"].min().date()} ~ {df_raw["date"].max().date()}')
df_raw.head(3)

---
## 2. モメンタム因子構築

In [ ]:
def log_return(series: pd.Series, window: int, skip: int = 1) -> pd.Series:
    """log return over `window` days, skipping `skip` most recent days."""
    return np.log(series.shift(skip) / series.shift(window + skip))


df = df_raw.copy()

# --- 短期リターン（反転期待）---
df['ret_1d'] = df.groupby('symbol')['close'].transform(lambda x: x.pct_change(1))
df['ret_5d'] = df.groupby('symbol')['close'].transform(lambda x: x.pct_change(5))

# --- モメンタム因子（スキップ1日）---
# skip=1: 最新日を除外（マイクロストラクチャーノイズ除去の標準的処理）
df['mom_1m']  = df.groupby('symbol')['close'].transform(lambda x: log_return(x, 21,  skip=1))
df['mom_3m']  = df.groupby('symbol')['close'].transform(lambda x: log_return(x, 63,  skip=1))
df['mom_6m']  = df.groupby('symbol')['close'].transform(lambda x: log_return(x, 126, skip=1))
df['mom_12m'] = df.groupby('symbol')['close'].transform(lambda x: log_return(x, 252, skip=1))

# --- 12m_minus_1m（標準的モメンタム: 1年リターンから直近1ヶ月を除く）---
df['mom_12m_minus_1m'] = df['mom_12m'] - df['mom_1m']

# --- フォワードリターン（ターゲット）---
df['fwd_ret_5d'] = df.groupby('symbol')['close'].transform(
    lambda x: x.pct_change(5).shift(-5)
)
df['fwd_ret_21d'] = df.groupby('symbol')['close'].transform(
    lambda x: x.pct_change(21).shift(-21)
)

# --- 月次データに集約（月末の値を代表値として使用）---
df['ym'] = df['date'].dt.to_period('M')

MOM_COLS = ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12m_minus_1m', 'ret_1d', 'ret_5d']
TARGET_COLS = ['fwd_ret_5d', 'fwd_ret_21d']

print('モメンタム因子構築完了')
print(f'全行数: {len(df):,}')
print(f'因子欠損率:')
for col in MOM_COLS:
    miss = df[col].isna().mean()
    print(f'  {col:25s}: {miss:.1%}')

In [ ]:
# 因子分布の確認
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

plot_cols = MOM_COLS + ['fwd_ret_5d']
for i, col in enumerate(plot_cols):
    data = df[col].dropna()
    # Winsorize at 1%/99%
    lo, hi = data.quantile([0.01, 0.99])
    data_w = data.clip(lo, hi)
    axes[i].hist(data_w, bins=50, color='steelblue', edgecolor='white', alpha=0.8, density=True)
    axes[i].axvline(0, color='red', linestyle='--', linewidth=1)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('値（上下1%ウィンソライズ）', fontsize=8)
    mu, sigma = data.mean(), data.std()
    axes[i].text(0.97, 0.97, f'μ={mu:.3f}\nσ={sigma:.3f}',
                 transform=axes[i].transAxes, va='top', ha='right', fontsize=8,
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 余剰axesを非表示
for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('モメンタム因子・ターゲット分布', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_mom_factor_dist.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. IC 分析（月次スピアマン相関）

Information Coefficient (IC) = 因子と将来リターンのスピアマン順位相関。  
IC > 0.03 が実用的目安。IC の t 値で H₀: IC=0 を検定する。

In [ ]:
from scipy.stats import spearmanr


def monthly_ic(df: pd.DataFrame, factor_col: str, target_col: str = 'fwd_ret_5d') -> pd.Series:
    """月末時点の因子値と翌5日リターンのスピアマンIC（月次）を計算する。"""
    # 月末の最終営業日を代表値として使用
    df_month = df.sort_values('date').groupby(['symbol', 'ym']).last().reset_index()
    result = {}
    for ym, grp in df_month.groupby('ym'):
        sub = grp[[factor_col, target_col]].dropna()
        if len(sub) < 10:  # 銘柄数が少ない月はスキップ
            continue
        ic, _ = spearmanr(sub[factor_col], sub[target_col])
        result[ym] = ic
    return pd.Series(result, name=factor_col)


# 各モメンタム因子のIC時系列を計算
ic_dict = {}
for col in MOM_COLS:
    ic_dict[col] = monthly_ic(df, col, 'fwd_ret_5d')
    print(f'{col:25s}: {len(ic_dict[col])} months computed')

ic_df = pd.DataFrame(ic_dict)
ic_df.index = ic_df.index.astype(str)
print(f'\nIC DataFrame shape: {ic_df.shape}')
ic_df.tail(3)

In [ ]:
# IC 時系列チャート（5つのモメンタム因子）
mom_factors = ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12m_minus_1m']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(len(mom_factors), 1, figsize=(14, 12), sharex=True)
x = np.arange(len(ic_df))
xtick_step = max(1, len(ic_df) // 12)

for i, (col, color) in enumerate(zip(mom_factors, colors)):
    ax = axes[i]
    vals = ic_df[col].values
    bar_colors = [color if v > 0 else '#aaaaaa' for v in vals]
    ax.bar(x, vals, color=bar_colors, alpha=0.7, width=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    mean_ic = np.nanmean(vals)
    ax.axhline(mean_ic, color=color, linewidth=1.5, linestyle='--',
               label=f'Mean IC = {mean_ic:+.4f}')
    ax.set_ylabel('IC', fontsize=9)
    ax.set_title(col, fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xticks(x[::xtick_step])
    ax.set_xticklabels(ic_df.index[::xtick_step], rotation=45, ha='right', fontsize=7)

plt.suptitle('モメンタム因子 月次 IC (vs fwd_ret_5d)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_mom_ic_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# IC 統計量テーブル（mean, std, t-stat, p-value, ICIR）
from scipy.stats import ttest_1samp

rows = []
for col in MOM_COLS:
    ic_vals = ic_df[col].dropna().values
    n = len(ic_vals)
    mean_ic = ic_vals.mean()
    std_ic = ic_vals.std(ddof=1)
    # t 検定: H₀: IC = 0
    t_stat, p_val = ttest_1samp(ic_vals, popmean=0)
    icir = mean_ic / std_ic if std_ic > 0 else np.nan  # IC / std(IC)
    pos_rate = (ic_vals > 0).mean()
    rows.append({
        '因子': col,
        'n_months': n,
        'IC_mean': mean_ic,
        'IC_std': std_ic,
        'ICIR': icir,
        't_stat': t_stat,
        'p_value': p_val,
        '有意(α=0.05)': '✓' if p_val < ALPHA else '✗',
        '正の月割合': pos_rate,
    })

ic_stats = pd.DataFrame(rows).set_index('因子')
print('=== IC 統計量サマリー ===')
print(ic_stats.to_string())

---
## 4. クインタイル・ポートフォリオ分析

`mom_12m_minus_1m` で毎月5分位に分類し、翌月リターンを集計する。

In [ ]:
# 月末時点のデータで分位分類
FACTOR = 'mom_12m_minus_1m'
TARGET = 'fwd_ret_21d'  # 翌月リターン

df_month = df.sort_values('date').groupby(['symbol', 'ym']).last().reset_index()

def assign_quintile(grp, factor_col, n=5):
    valid = grp[factor_col].notna()
    grp = grp.copy()
    grp.loc[valid, 'quintile'] = pd.qcut(
        grp.loc[valid, factor_col], q=n,
        labels=[f'Q{i+1}' for i in range(n)],
        duplicates='drop'
    )
    return grp

df_month = df_month.groupby('ym', group_keys=False).apply(
    lambda g: assign_quintile(g, FACTOR)
)

# 分位別リターン集計
quintile_ret = (
    df_month.dropna(subset=['quintile', TARGET])
    .groupby(['ym', 'quintile'])[TARGET]
    .mean()
    .unstack('quintile')
)
quintile_ret.index = quintile_ret.index.astype(str)

# Long-Short スプレッド
if 'Q5' in quintile_ret.columns and 'Q1' in quintile_ret.columns:
    quintile_ret['LS'] = quintile_ret['Q5'] - quintile_ret['Q1']

print('クインタイル別月次リターン (先頭5行):')
print(quintile_ret.head())
print(f'\n期間: {quintile_ret.index[0]} ~ {quintile_ret.index[-1]}')

In [ ]:
# 累積リターンチャート
cum_ret = (1 + quintile_ret).cumprod() - 1

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Q1 / Q3 / Q5 + LS
palette = {'Q1': '#d62728', 'Q3': '#7f7f7f', 'Q5': '#2ca02c', 'LS': '#1f77b4'}
ax = axes[0]
for col in ['Q1', 'Q3', 'Q5', 'LS']:
    if col in cum_ret.columns:
        ax.plot(cum_ret.index, cum_ret[col] * 100,
                label=col, color=palette.get(col), linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'クインタイル累積リターン ({FACTOR})', fontsize=12)
ax.set_ylabel('累積リターン (%)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
step = max(1, len(cum_ret) // 12)
ax.set_xticks(range(0, len(cum_ret), step))
ax.set_xticklabels(cum_ret.index[::step], rotation=45, ha='right', fontsize=7)

# 全クインタイル
ax2 = axes[1]
q_cols = [c for c in cum_ret.columns if c.startswith('Q')]
for col in q_cols:
    ax2.plot(cum_ret.index, cum_ret[col] * 100, label=col, linewidth=1.2)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('全クインタイル累積リターン（単調性の確認）', fontsize=12)
ax2.set_ylabel('累積リターン (%)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.set_xticks(range(0, len(cum_ret), step))
ax2.set_xticklabels(cum_ret.index[::step], rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_quintile_cumret.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# クインタイル別パフォーマンス指標
def calc_performance(ret_series: pd.Series, periods_per_year: int = 12) -> dict:
    """年率リターン・シャープ・最大ドローダウンを計算する（月次リターン想定）。"""
    ret = ret_series.dropna()
    ann_ret = (1 + ret.mean()) ** periods_per_year - 1
    ann_vol = ret.std() * np.sqrt(periods_per_year)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    # 最大ドローダウン
    cum = (1 + ret).cumprod()
    peak = cum.expanding().max()
    dd = (cum - peak) / peak
    max_dd = dd.min()
    return {
        '年率リターン': f'{ann_ret:.1%}',
        '年率ボラ': f'{ann_vol:.1%}',
        'シャープ比': f'{sharpe:.2f}',
        '最大DD': f'{max_dd:.1%}',
    }

perf_rows = []
for col in [c for c in quintile_ret.columns if c.startswith('Q')] + ['LS']:
    if col in quintile_ret.columns:
        row = {'ポートフォリオ': col}
        row.update(calc_performance(quintile_ret[col]))
        perf_rows.append(row)

perf_df = pd.DataFrame(perf_rows).set_index('ポートフォリオ')
print('=== クインタイル別パフォーマンス ===')
print(perf_df.to_string())

---
## 5. 市場レジーム別クロスマーケット比較

高ボラ期 vs 低ボラ期、上昇相場 vs 下落相場で IC の安定性を確認する。

In [ ]:
# 市場ボラティリティレジームの判定
# ニッケイ225 の実現ボラティリティでレジームを定義

if df_macro is not None and 'nikkei225' in df_macro.columns:
    mkt = df_macro.set_index('date')['nikkei225'].copy()
    mkt_ret = np.log(mkt / mkt.shift(1))
    mkt_vol = mkt_ret.rolling(21).std() * np.sqrt(252)  # 月次実現ボラ
    mkt_cum = mkt / mkt.iloc[0] - 1
    mkt_df = pd.DataFrame({
        'date': mkt.index,
        'nikkei': mkt.values,
        'mkt_vol': mkt_vol.values,
        'mkt_ret': mkt_ret.values,
    })
    mkt_df['ym'] = pd.to_datetime(mkt_df['date']).dt.to_period('M')

    # 月次レジーム集計
    regime_monthly = mkt_df.groupby('ym').agg(
        mkt_vol_mean=('mkt_vol', 'mean'),
        nikkei_ret=('mkt_ret', 'sum'),
    ).reset_index()
    vol_median = regime_monthly['mkt_vol_mean'].median()
    regime_monthly['vol_regime'] = np.where(
        regime_monthly['mkt_vol_mean'] > vol_median, 'High_Vol', 'Low_Vol'
    )
    regime_monthly['mkt_regime'] = np.where(
        regime_monthly['nikkei_ret'] > 0, 'Up_Market', 'Down_Market'
    )
    regime_monthly['ym'] = regime_monthly['ym'].astype(str)

    # IC とレジームを結合
    ic_with_regime = ic_df.copy()
    ic_with_regime.index.name = 'ym'
    ic_with_regime = ic_with_regime.reset_index()
    ic_with_regime = ic_with_regime.merge(regime_monthly, on='ym', how='inner')

    # レジーム別IC比較
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, regime_col, title in zip(
        axes,
        ['vol_regime', 'mkt_regime'],
        ['ボラティリティレジーム別 IC', '市場方向別 IC']
    ):
        regime_ic = ic_with_regime.groupby(regime_col)[['mom_12m_minus_1m', 'mom_1m']].mean()
        regime_ic.plot(kind='bar', ax=ax, color=['#2ca02c', '#d62728'], alpha=0.8, edgecolor='white')
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(title, fontsize=11)
        ax.set_ylabel('平均 IC')
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=0)
        ax.legend(['mom_12m_minus_1m', 'mom_1m'])
        ax.grid(alpha=0.3, axis='y')

    plt.suptitle('レジーム別モメンタム IC', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '22_regime_ic.png', dpi=120, bbox_inches='tight')
    plt.show()

    print('\n=== レジーム別 IC 集計 ===')
    for regime_col in ['vol_regime', 'mkt_regime']:
        print(f'\n{regime_col}:')
        print(ic_with_regime.groupby(regime_col)[['mom_12m_minus_1m', 'mom_3m', 'mom_1m']]
              .agg(['mean', 'std', 'count']).round(4))
else:
    print('マクロデータが利用できないため、レジーム分析をスキップします。')
    print('macro.parquet に nikkei225 列が必要です。')

---
## 6. 統計的頑健性検定

In [ ]:
# ---- 6-1. ブートストラップ IC 信頼区間（1000 反復）----
FACTOR_COL = 'mom_12m_minus_1m'
N_BOOT = 1000

ic_vals = ic_df[FACTOR_COL].dropna().values

boot_means = np.array([
    np.mean(np.random.choice(ic_vals, size=len(ic_vals), replace=True))
    for _ in range(N_BOOT)
])

ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])
observed_mean = ic_vals.mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_means, bins=40, color='steelblue', edgecolor='white', alpha=0.8, density=True)
ax.axvline(observed_mean, color='red', linewidth=2, label=f'Observed Mean = {observed_mean:.4f}')
ax.axvline(ci_lower, color='orange', linewidth=1.5, linestyle='--', label=f'95% CI Lower = {ci_lower:.4f}')
ax.axvline(ci_upper, color='orange', linewidth=1.5, linestyle='--', label=f'95% CI Upper = {ci_upper:.4f}')
ax.axvline(0, color='black', linewidth=1)
ax.set_title(f'ブートストラップ IC 分布 ({FACTOR_COL}, n={N_BOOT})', fontsize=11)
ax.set_xlabel('Bootstrap Mean IC')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_bootstrap_ic.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'=== ブートストラップ信頼区間 ({FACTOR_COL}) ===')
print(f'観測 IC 平均: {observed_mean:+.4f}')
print(f'95% CI : [{ci_lower:+.4f}, {ci_upper:+.4f}]')
print(f'H₀ (IC=0) 棄却: {"Yes" if ci_lower > 0 or ci_upper < 0 else "No"}')

In [ ]:
# ---- 6-2. サブ期間分析 ----
subperiods = {
    '2020-21': ('2020-01', '2021-12'),
    '2022-23': ('2022-01', '2023-12'),
    '2024-25': ('2024-01', '2025-12'),
}

print('=== サブ期間別 IC 分析 ===')
sub_rows = []
for period_name, (start, end) in subperiods.items():
    mask = (ic_df.index >= start) & (ic_df.index <= end)
    if mask.sum() < 3:
        print(f'  {period_name}: データ不足 ({mask.sum()} months)')
        continue
    for col in ['mom_12m_minus_1m', 'mom_3m', 'mom_1m']:
        sub_vals = ic_df.loc[mask, col].dropna().values
        if len(sub_vals) < 3:
            continue
        t_stat, p_val = ttest_1samp(sub_vals, popmean=0)
        sub_rows.append({
            '期間': period_name,
            '因子': col,
            'n': len(sub_vals),
            'IC_mean': sub_vals.mean(),
            'IC_std': sub_vals.std(),
            't_stat': t_stat,
            'p_value': p_val,
            '有意': '✓' if p_val < ALPHA else '✗',
        })

if sub_rows:
    sub_df = pd.DataFrame(sub_rows)
    print(sub_df.to_string(index=False))
else:
    print('サブ期間データが不足しています。データ収集期間を確認してください。')

In [ ]:
# ---- 6-3. 置換検定（Permutation Test）----
N_PERM = 1000

# 月次に整形した因子・ターゲットペア
df_perm = df.sort_values('date').groupby(['symbol', 'ym']).last().reset_index()
df_perm = df_perm[['ym', FACTOR_COL, 'fwd_ret_5d']].dropna()

if len(df_perm) > 0:
    # 観測IC（全サンプル版）
    obs_ic, _ = spearmanr(df_perm[FACTOR_COL], df_perm['fwd_ret_5d'])

    # 置換版
    perm_ics = []
    targets = df_perm['fwd_ret_5d'].values.copy()
    factors = df_perm[FACTOR_COL].values.copy()
    for _ in range(N_PERM):
        shuffled = np.random.permutation(targets)
        ic_perm, _ = spearmanr(factors, shuffled)
        perm_ics.append(ic_perm)
    perm_ics = np.array(perm_ics)

    p_perm = (np.abs(perm_ics) >= np.abs(obs_ic)).mean()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(perm_ics, bins=40, color='gray', edgecolor='white', alpha=0.7,
            density=True, label='Permuted IC')
    ax.axvline(obs_ic, color='red', linewidth=2, label=f'Observed IC = {obs_ic:.4f}')
    ax.set_title(f'置換検定: {FACTOR_COL} vs fwd_ret_5d', fontsize=11)
    ax.set_xlabel('IC')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '22_permutation_test.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f'=== 置換検定結果 ===')
    print(f'観測 IC: {obs_ic:+.4f}')
    print(f'置換検定 p値: {p_perm:.4f}')
    print(f'H₀ 棄却: {"Yes" if p_perm < ALPHA else "No"}  (α={ALPHA})')
else:
    print('置換検定に十分なデータがありません。')

---
## 7. 短期反転分析

短期（1日・5日）リターンは将来リターンに対して**負の IC**（反転）が期待される。

In [ ]:
# 短期反転因子の IC
reversal_cols = ['ret_1d', 'ret_5d']

rev_ic_dict = {}
for col in reversal_cols:
    rev_ic_dict[col] = monthly_ic(df, col, 'fwd_ret_5d')

rev_ic_df = pd.DataFrame(rev_ic_dict)
rev_ic_df.index = rev_ic_df.index.astype(str)

# IC テーブル
print('=== 短期反転因子 IC ===')
rev_rows = []
for col in reversal_cols:
    ic_vals = rev_ic_df[col].dropna().values
    t_stat, p_val = ttest_1samp(ic_vals, popmean=0)
    rev_rows.append({
        '因子': col,
        'IC_mean': ic_vals.mean(),
        'IC_std': ic_vals.std(),
        't_stat': t_stat,
        'p_value': p_val,
        '反転期待通り': '✓ (IC < 0)' if ic_vals.mean() < 0 else '✗ (IC ≥ 0)',
    })
print(pd.DataFrame(rev_rows).to_string(index=False))

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, reversal_cols):
    vals = rev_ic_df[col].values
    bar_colors = ['#2ca02c' if v > 0 else '#d62728' for v in vals]
    ax.bar(range(len(vals)), vals, color=bar_colors, alpha=0.7, width=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    mean_ic = np.nanmean(vals)
    ax.axhline(mean_ic, color='orange', linewidth=1.5, linestyle='--',
               label=f'Mean IC = {mean_ic:+.4f}')
    ax.set_title(f'短期反転: {col}', fontsize=11)
    ax.set_ylabel('月次 IC')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('短期反転因子 IC（負のICが反転効果を示す）', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_reversal_ic.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# モメンタム × 反転 オーバーレイ
# 「モメンタムが強い期間に短期反転も強い/弱い?」を検証

if len(ic_df) > 0 and len(rev_ic_df) > 0:
    # 共通インデックスで結合
    combined = pd.concat([
        ic_df['mom_12m_minus_1m'].rename('mom_IC'),
        rev_ic_df['ret_5d'].rename('rev_IC')
    ], axis=1).dropna()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 散布図
    axes[0].scatter(combined['mom_IC'], combined['rev_IC'], alpha=0.5, s=30, color='steelblue')
    # 回帰直線
    slope, intercept, r, p, se = stats.linregress(combined['mom_IC'], combined['rev_IC'])
    x_line = np.linspace(combined['mom_IC'].min(), combined['mom_IC'].max(), 100)
    axes[0].plot(x_line, intercept + slope * x_line, 'r-', linewidth=1.5,
                 label=f'r={r:.3f}, p={p:.3f}')
    axes[0].axhline(0, color='black', linewidth=0.5)
    axes[0].axvline(0, color='black', linewidth=0.5)
    axes[0].set_xlabel('mom_12m_minus_1m IC')
    axes[0].set_ylabel('ret_5d IC（反転）')
    axes[0].set_title('モメンタム IC vs 短期反転 IC')
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    # 時系列オーバーレイ
    x = np.arange(len(combined))
    ax2 = axes[1]
    ax2.plot(x, combined['mom_IC'].values, label='mom_12m_minus_1m IC', color='#1f77b4', linewidth=1.2)
    ax_twin = ax2.twinx()
    ax_twin.plot(x, combined['rev_IC'].values, label='ret_5d IC（反転）',
                 color='#d62728', linewidth=1.2, linestyle='--')
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.set_ylabel('モメンタム IC', color='#1f77b4')
    ax_twin.set_ylabel('反転 IC', color='#d62728')
    ax2.set_title('モメンタム IC と反転 IC の時系列比較')
    step = max(1, len(combined) // 8)
    ax2.set_xticks(x[::step])
    ax2.set_xticklabels(combined.index[::step], rotation=45, ha='right', fontsize=7)
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax_twin.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
    ax2.grid(alpha=0.3)

    plt.suptitle('モメンタム × 反転 インタラクション', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '22_mom_reversal_interaction.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## 8. 結論

### 仮説検定結果サマリー

| 仮説 | 内容 |
|------|------|
| **H₀** | IC = 0（モメンタム効果なし） |
| **H₁** | IC ≠ 0（モメンタム効果あり） |
| **有意水準** | α = 0.05 |

### 検定結果
<!-- 実行後に記入 -->

| 因子 | IC 平均 | t 値 | p 値 | 判定 |
|------|---------|------|------|------|
| mom_12m_minus_1m | — | — | — | — |
| mom_3m | — | — | — | — |
| mom_1m（反転） | — | — | — | — |

### 効果量
- ICIR = IC_mean / IC_std で評価。ICIR > 0.3 で実用的に有効とされる

### トレーディング上の含意
1. **モメンタム効果が確認された場合**: `mom_12m_minus_1m` を LightGBM の特徴量として採用する。クインタイル分析で Q5−Q1 のスプレッドが正であることを確認する
2. **短期反転が有意な場合**: 最新1日・5日リターンに負のウェイトを与える、または直近リターンでの調整を実施する
3. **レジーム依存性が強い場合**: レジームフィルタ（vol_regime, mkt_regime）を特徴量として追加し、モデルに学習させる

### 次のステップ
- `23_pead_hypothesis.ipynb` で PEAD 効果を検証する
- `42_equity_lgbm_walkforward.ipynb` でモメンタム因子を特徴量セットに組み込む

In [ ]:
# 最終サマリーテーブルの出力と保存
print('=== 最終サマリー: モメンタム因子 IC 統計 ===')
print(ic_stats.to_string())

# CSV 保存
ic_stats.to_csv(TABLES_DIR / '22_momentum_ic_stats.csv', encoding='utf-8-sig')
ic_df.to_csv(TABLES_DIR / '22_momentum_ic_timeseries.csv', encoding='utf-8-sig')
print(f'\n保存: {TABLES_DIR}/22_momentum_ic_stats.csv')
print(f'保存: {TABLES_DIR}/22_momentum_ic_timeseries.csv')